# 03 - Fine-Tuned Model Comparison

This notebook compares our fine-tuned BioBERT model against the baseline pretrained model.

**Purpose:**
- Evaluate the fine-tuned model on the same realistic user inputs from notebook 02
- Compare metrics (precision, recall, F1) before and after fine-tuning
- Analyze improvements and remaining weaknesses

In [1]:
import warnings
warnings.filterwarnings('ignore')

import sys
import os
from pathlib import Path

# Add project root to path
project_root = Path().absolute().parent
sys.path.insert(0, str(project_root))

import torch
import json
from collections import defaultdict
from datetime import datetime
import pandas as pd

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'}")

PyTorch version: 2.9.1
Device: mps


## 1. Load the Fine-Tuned Model

In [4]:
# Find the latest trained model
model_dir = project_root / "outputs" / "models"
model_runs = sorted(model_dir.glob("run_*"))

if model_runs:
    latest_model = model_runs[-1] / "final_model"
    print(f"Using model: {latest_model}")
else:
    raise FileNotFoundError("No trained models found. Run train.py first!")

# Load training info
training_info_path = model_runs[-1] / "training_info.json"
if training_info_path.exists():
    with open(training_info_path) as f:
        training_info = json.load(f)
    print(f"\nTraining Info:")
    print(f"  Epochs: {training_info.get('config', {}).get('epochs', 'N/A')}")
    f1_score = training_info.get('final_metrics', {}).get('f1', 'N/A')
    if isinstance(f1_score, (int, float)):
        print(f"  Final F1: {f1_score:.4f}")
    else:
        print(f"  Final F1: {f1_score}")

Using model: /Users/ahmedmusayev/Desktop/AI:ML:DL/projects/medai/outputs/models/run_20260130_144730/final_model

Training Info:
  Epochs: N/A
  Final F1: N/A


In [8]:
# Import prediction module
from src.predict import NERPredictor

# Initialize the predictor
print("Loading fine-tuned model...")
predictor = NERPredictor(str(latest_model))
print("Model loaded!")

Loading fine-tuned model...
Import error: cannot import name 'PreTrainedModel' from 'transformers' (/Users/ahmedmusayev/anaconda3/lib/python3.11/site-packages/transformers/__init__.py)

Trying to load model directly without PreTrainedModel type hints...


ImportError: cannot import name 'PreTrainedModel' from 'transformers' (/Users/ahmedmusayev/anaconda3/lib/python3.11/site-packages/transformers/__init__.py)

## 2. Define Same User Inputs from Baseline Evaluation

In [9]:
# Same realistic user inputs from notebook 02
USER_INPUTS = [
    # Simple symptom descriptions
    "I have a headache and feel tired all the time.",
    "My throat hurts and I have a runny nose.",
    "I've been having chest pain for the past 2 days.",
    "I feel dizzy when I stand up quickly.",
    "My stomach has been hurting after eating.",
    
    # More complex descriptions
    "I woke up with a fever, body aches, and chills.",
    "I have trouble sleeping and my heart races at night.",
    "My joints are swollen and painful, especially in the morning.",
    "I've noticed shortness of breath when climbing stairs.",
    "I have a persistent cough that won't go away.",
    
    # Casual/conversational style
    "Been feeling really nauseous lately, can't keep food down.",
    "My back is killing me, especially the lower back.",
    "Got this weird rash on my arm that's super itchy.",
    "Can't stop sneezing and my eyes are watery.",
    "Feeling anxious and having trouble concentrating.",
    
    # With medications mentioned
    "I took ibuprofen for my migraine but it didn't help.",
    "I've been on aspirin for my heart, but now I have stomach issues.",
    "The doctor gave me amoxicillin for my infection.",
    
    # With conditions mentioned
    "I have diabetes and lately my vision has been blurry.",
    "My asthma has been acting up, I'm wheezing a lot.",
    "I was diagnosed with hypertension last year.",
    
    # Edge cases - vague descriptions
    "I just don't feel well.",
    "Something is wrong but I can't explain it.",
    "I feel off today.",
    
    # Multiple symptoms
    "I have a high fever, severe headache, stiff neck, and sensitivity to light.",
    "Experiencing fatigue, weight loss, increased thirst, and frequent urination.",
]

print(f"Total test inputs: {len(USER_INPUTS)}")

Total test inputs: 26


## 3. Run Fine-Tuned Model Evaluation

In [10]:
print("Running fine-tuned model evaluation on all user inputs...\n")
print("=" * 80)

finetuned_results = []
entity_type_counts = defaultdict(int)
total_entities = 0

for i, text in enumerate(USER_INPUTS, 1):
    # Get predictions from fine-tuned model
    prediction = predictor.predict(text)
    entities = prediction.get('entities', [])
    
    result = {
        'input': text,
        'entities': entities,
        'entity_count': len(entities)
    }
    finetuned_results.append(result)
    
    # Count entity types
    for ent in entities:
        entity_type_counts[ent.get('label', 'Unknown')] += 1
        total_entities += 1
    
    # Print results
    print(f"\n[{i}/{len(USER_INPUTS)}] Input: {text}")
    if entities:
        for ent in entities:
            conf = ent.get('confidence', 0)
            print(f"    -> {ent['text']}: {ent['label']} (conf: {conf:.3f})")
    else:
        print("    -> No entities detected")

print("\n" + "=" * 80)

Running fine-tuned model evaluation on all user inputs...



NameError: name 'predictor' is not defined

## 4. Calculate Fine-Tuned Model Metrics

In [ ]:
# Expected entities (same as baseline notebook)
EXPECTED_ENTITIES = {
    "I have a headache and feel tired all the time.": ["headache", "tired"],
    "My throat hurts and I have a runny nose.": ["throat hurts", "runny nose"],
    "I've been having chest pain for the past 2 days.": ["chest pain"],
    "I feel dizzy when I stand up quickly.": ["dizzy"],
    "My stomach has been hurting after eating.": ["stomach hurting"],
    "I woke up with a fever, body aches, and chills.": ["fever", "body aches", "chills"],
    "I have trouble sleeping and my heart races at night.": ["trouble sleeping", "heart races"],
    "My joints are swollen and painful, especially in the morning.": ["joints swollen", "painful"],
    "I've noticed shortness of breath when climbing stairs.": ["shortness of breath"],
    "I have a persistent cough that won't go away.": ["cough"],
    "Been feeling really nauseous lately, can't keep food down.": ["nauseous"],
    "My back is killing me, especially the lower back.": ["back pain", "lower back"],
    "Got this weird rash on my arm that's super itchy.": ["rash", "itchy"],
    "Can't stop sneezing and my eyes are watery.": ["sneezing", "watery eyes"],
    "Feeling anxious and having trouble concentrating.": ["anxious", "trouble concentrating"],
    "I took ibuprofen for my migraine but it didn't help.": ["ibuprofen", "migraine"],
    "I've been on aspirin for my heart, but now I have stomach issues.": ["aspirin", "stomach issues"],
    "The doctor gave me amoxicillin for my infection.": ["amoxicillin", "infection"],
    "I have diabetes and lately my vision has been blurry.": ["diabetes", "blurry vision"],
    "My asthma has been acting up, I'm wheezing a lot.": ["asthma", "wheezing"],
    "I was diagnosed with hypertension last year.": ["hypertension"],
}

# Calculate approximate precision/recall
total_expected = 0
total_detected = 0
total_correct = 0

for result in finetuned_results:
    text = result['input']
    if text in EXPECTED_ENTITIES:
        expected = EXPECTED_ENTITIES[text]
        detected = [e['text'].lower() for e in result['entities']]
        
        total_expected += len(expected)
        total_detected += len(detected)
        
        # Count correct detections (fuzzy matching)
        for exp in expected:
            for det in detected:
                if exp.lower() in det or det in exp.lower():
                    total_correct += 1
                    break

# Calculate metrics
ft_precision = total_correct / total_detected if total_detected > 0 else 0
ft_recall = total_correct / total_expected if total_expected > 0 else 0
ft_f1 = 2 * (ft_precision * ft_recall) / (ft_precision + ft_recall) if (ft_precision + ft_recall) > 0 else 0

print("\n" + "=" * 50)
print("FINE-TUNED MODEL METRICS (Approximate)")
print("=" * 50)
print(f"\nTotal expected entities: {total_expected}")
print(f"Total detected entities: {total_detected}")
print(f"Correctly detected: {total_correct}")
print(f"\nPrecision: {ft_precision:.2%}")
print(f"Recall: {ft_recall:.2%}")
print(f"F1 Score: {ft_f1:.2%}")

## 5. Load Baseline Results for Comparison

In [ ]:
# Try to load baseline results
baseline_path = project_root / "results" / "baseline_evaluation.json"

baseline_metrics = None
if baseline_path.exists():
    with open(baseline_path) as f:
        baseline_data = json.load(f)
    baseline_metrics = baseline_data.get('metrics', {})
    print("Loaded baseline results from results/baseline_evaluation.json")
    print(f"\nBaseline Model: {baseline_data.get('model', 'Unknown')}")
    print(f"Baseline Precision: {baseline_metrics.get('precision', 0):.2%}")
    print(f"Baseline Recall: {baseline_metrics.get('recall', 0):.2%}")
    print(f"Baseline F1: {baseline_metrics.get('f1', 0):.2%}")
else:
    print("No baseline results found. Run notebook 02 first to establish baseline.")
    # Use approximate baseline values from typical pretrained models
    baseline_metrics = {'precision': 0.40, 'recall': 0.35, 'f1': 0.37}

## 6. Compare Baseline vs Fine-Tuned

In [ ]:
print("\n" + "#" * 60)
print("#" + " COMPARISON: BASELINE vs FINE-TUNED ".center(58) + "#")
print("#" * 60)

if baseline_metrics:
    b_precision = baseline_metrics.get('precision', 0)
    b_recall = baseline_metrics.get('recall', 0)
    b_f1 = baseline_metrics.get('f1', 0)
    
    print(f"\n{'Metric':<15} {'Baseline':>12} {'Fine-Tuned':>12} {'Change':>12}")
    print("-" * 55)
    
    # Precision
    p_change = ft_precision - b_precision
    p_arrow = '+' if p_change >= 0 else ''
    print(f"{'Precision':<15} {b_precision:>11.2%} {ft_precision:>11.2%} {p_arrow}{p_change:>10.2%}")
    
    # Recall  
    r_change = ft_recall - b_recall
    r_arrow = '+' if r_change >= 0 else ''
    print(f"{'Recall':<15} {b_recall:>11.2%} {ft_recall:>11.2%} {r_arrow}{r_change:>10.2%}")
    
    # F1
    f1_change = ft_f1 - b_f1
    f1_arrow = '+' if f1_change >= 0 else ''
    print(f"{'F1 Score':<15} {b_f1:>11.2%} {ft_f1:>11.2%} {f1_arrow}{f1_change:>10.2%}")
    
    print("\n" + "#" * 60)

In [ ]:
# Visualize comparison
import matplotlib.pyplot as plt

if baseline_metrics:
    metrics = ['Precision', 'Recall', 'F1 Score']
    baseline_values = [b_precision, b_recall, b_f1]
    finetuned_values = [ft_precision, ft_recall, ft_f1]
    
    x = range(len(metrics))
    width = 0.35
    
    fig, ax = plt.subplots(figsize=(10, 6))
    bars1 = ax.bar([i - width/2 for i in x], baseline_values, width, label='Baseline (d4data)', color='#e74c3c', alpha=0.8)
    bars2 = ax.bar([i + width/2 for i in x], finetuned_values, width, label='Fine-Tuned BioBERT', color='#2ecc71', alpha=0.8)
    
    ax.set_ylabel('Score')
    ax.set_title('Baseline vs Fine-Tuned Model Comparison')
    ax.set_xticks(x)
    ax.set_xticklabels(metrics)
    ax.legend()
    ax.set_ylim(0, 1.1)
    
    # Add value labels
    for bar in bars1:
        height = bar.get_height()
        ax.annotate(f'{height:.1%}', xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3), textcoords="offset points", ha='center', va='bottom')
    for bar in bars2:
        height = bar.get_height()
        ax.annotate(f'{height:.1%}', xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3), textcoords="offset points", ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()

## 7. Entity Type Distribution

In [ ]:
print("\nEntity Type Distribution (Fine-Tuned Model):")
print("-" * 40)
for entity_type, count in sorted(entity_type_counts.items(), key=lambda x: -x[1]):
    percentage = (count / total_entities) * 100 if total_entities > 0 else 0
    print(f"  {entity_type}: {count} ({percentage:.1f}%)")
print(f"\nTotal entities detected: {total_entities}")
print(f"Average entities per input: {total_entities/len(USER_INPUTS):.2f}")

## 8. Save Fine-Tuned Results

In [ ]:
# Save fine-tuned results
finetuned_evaluation = {
    'timestamp': datetime.now().isoformat(),
    'model': str(latest_model),
    'model_type': 'finetuned_biobert',
    'metrics': {
        'precision': ft_precision,
        'recall': ft_recall,
        'f1': ft_f1,
        'total_inputs': len(USER_INPUTS),
        'total_entities_detected': total_entities,
        'avg_entities_per_input': total_entities / len(USER_INPUTS)
    },
    'entity_type_distribution': dict(entity_type_counts),
    'comparison': {
        'baseline_f1': b_f1 if baseline_metrics else None,
        'finetuned_f1': ft_f1,
        'improvement': ft_f1 - b_f1 if baseline_metrics else None
    },
    'detailed_results': finetuned_results
}

# Save to file
results_dir = project_root / "results"
results_dir.mkdir(exist_ok=True)

with open(results_dir / 'finetuned_evaluation.json', 'w') as f:
    json.dump(finetuned_evaluation, f, indent=2)

print("Fine-tuned results saved to results/finetuned_evaluation.json")

## 9. Summary

In [ ]:
print("\n" + "#" * 60)
print("#" + " EVALUATION COMPLETE ".center(58) + "#")
print("#" * 60)
print(f"\n  Fine-Tuned Model: {latest_model.name}")
print(f"  Test inputs: {len(USER_INPUTS)}")
print(f"  Entities detected: {total_entities}")
print(f"\n  Fine-Tuned Metrics:")
print(f"    Precision: {ft_precision:.2%}")
print(f"    Recall:    {ft_recall:.2%}")
print(f"    F1 Score:  {ft_f1:.2%}")

if baseline_metrics:
    print(f"\n  Improvement over Baseline:")
    print(f"    F1 Score: {b_f1:.2%} -> {ft_f1:.2%} ({'+' if f1_change >= 0 else ''}{f1_change:.2%})")

print("\n" + "#" * 60)